# 02 — Dataset Preparation, Validation & Drive Persistence
# Chuẩn bị Dữ liệu, Kiểm định Chất lượng & Lưu trữ lên Drive

**English:**
This notebook runs `prepare_dataset.py` and `validate_dataset.py` on `e:\mimamori\ai-training`, inspects category balance across all 8 required categories, confirms zero split/benchmark leakage and zero PII, optionally inspects token lengths using the `Qwen/Qwen3-4B-Instruct-2507` tokenizer, and persists processed splits to Google Drive (`/content/drive/MyDrive/mimamori_companion_ai/processed_data`).

**Tiếng Việt:**
Notebook này chạy `prepare_dataset.py` và `validate_dataset.py`, kiểm tra độ cân bằng của đủ 8 nhóm chủ đề, xác minh không rò rỉ dữ liệu giữa train/val/test và tập đánh giá (`zero_leakage`), kiểm tra không chứa thông tin nhạy cảm (PII), tùy chọn thống kê độ dài token theo tokenizer của Qwen3, và sao lưu các tập dữ liệu đã xử lý lên Google Drive.

> **Synthetic Seed Warning / Cảnh báo dữ liệu mẫu:**
> - `datasets/raw/synthetic_companion_seed.jsonl` (`source: synthetic_mimamori_seed_v1`, 32 dialogues) and `datasets/evaluation/eval_benchmark.jsonl` (`source: synthetic_eval_benchmark_v1`, 20 cases) have `reviewed: false`.
> - They are intended **only** for pipeline verification and smoke testing until clinically/culturally reviewed (`reviewed: true`).


## Step 1: Mount Drive, Load Configs & Check T4 Hardware Safeguards
## Bước 1: Kết nối Drive, Đọc Cấu hình & Kiểm tra Giới hạn T4


In [ ]:
import os
import sys
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT_DEFAULT = Path("/content/drive/MyDrive/mimamori_companion_ai")

# Optional: set AI_TRAINING_DIR explicitly if your repo lives in a custom folder
CUSTOM_AI_TRAINING_DIR = os.environ.get("MIMAMORI_AI_TRAINING_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("[Drive] Google Drive mounted at /content/drive")
else:
    print("[Local] Not running inside Google Colab; Drive mount skipped.")


def resolve_ai_training_dir() -> Path:
    candidates: list[Path] = []
    if CUSTOM_AI_TRAINING_DIR:
        candidates.append(Path(CUSTOM_AI_TRAINING_DIR))
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / "ai-training",
        cwd.parent / "ai-training",
        Path("/content/mimamori/ai-training"),
        Path("/content/ai-training"),
        Path("/content/drive/MyDrive/mimamori/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/repo/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/ai-training"),
    ])
    for cand in candidates:
        if (cand / "configs" / "model_config.yaml").exists() and (cand / "configs" / "training_config.yaml").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Could not locate 'ai-training/configs/model_config.yaml'. "
        "In Colab, clone/upload the repository to /content/mimamori or Google Drive, "
        "or set CUSTOM_AI_TRAINING_DIR / MIMAMORI_AI_TRAINING_DIR."
    )


AI_TRAINING_DIR = resolve_ai_training_dir()
if str(AI_TRAINING_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(AI_TRAINING_DIR / "scripts"))

import yaml

MODEL_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "model_config.yaml"
TRAINING_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "training_config.yaml"
with MODEL_CONFIG_PATH.open("r", encoding="utf-8") as f:
    MODEL_CFG = yaml.safe_load(f)
with TRAINING_CONFIG_PATH.open("r", encoding="utf-8") as f:
    TRAIN_CFG = yaml.safe_load(f)

PATHS_CFG = TRAIN_CFG["paths"]
if IN_COLAB:
    DRIVE_ROOT = Path(PATHS_CFG["drive_root"])
    DRIVE_CHECKPOINTS_DIR = Path(PATHS_CFG["drive_checkpoints_dir"])
    DRIVE_FINAL_ADAPTER_DIR = Path(PATHS_CFG["drive_final_adapter_dir"])
    DRIVE_EVAL_OUTPUTS_DIR = Path(PATHS_CFG["drive_eval_outputs_dir"])
    DRIVE_PROCESSED_DATA_DIR = Path(PATHS_CFG["drive_processed_data_dir"])
else:
    local_drive_sim = AI_TRAINING_DIR / "outputs" / "_local_drive_mirror"
    DRIVE_ROOT = local_drive_sim
    DRIVE_CHECKPOINTS_DIR = local_drive_sim / "checkpoints"
    DRIVE_FINAL_ADAPTER_DIR = local_drive_sim / "adapters" / "qwen3-4b-companion-lora"
    DRIVE_EVAL_OUTPUTS_DIR = AI_TRAINING_DIR / PATHS_CFG["local_outputs_dir"]
    DRIVE_PROCESSED_DATA_DIR = AI_TRAINING_DIR / "datasets" / "processed"

for d in (DRIVE_ROOT, DRIVE_CHECKPOINTS_DIR, DRIVE_FINAL_ADAPTER_DIR, DRIVE_EVAL_OUTPUTS_DIR, DRIVE_PROCESSED_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"[Repo] AI_TRAINING_DIR      = {AI_TRAINING_DIR}")
print(f"[Drive] DRIVE_ROOT          = {DRIVE_ROOT}")
print(f"[Drive] DRIVE_EVAL_OUTPUTS  = {DRIVE_EVAL_OUTPUTS_DIR}")
print(f"[Drive] DRIVE_PROCESSED     = {DRIVE_PROCESSED_DATA_DIR}")


def inspect_gpu_and_t4_safeguards(train_cfg: dict) -> dict:
    """Detect GPU/VRAM and return Free Colab T4 15GB safe hyperparameters."""
    t4_target = train_cfg["hardware_target"]
    base_train = dict(train_cfg["training"])
    fallback = dict(train_cfg["t4_oom_fallback"])
    info = {
        "cuda_available": False,
        "gpu_name": "CPU",
        "vram_gb": 0.0,
        "compute_capability": None,
        "supports_bf16": False,
        "supports_flash_attention_2": False,
        "use_t4_oom_fallback": False,
        "effective_fp16": True,
        "effective_bf16": False,
        "effective_batch_size": fallback["per_device_train_batch_size"],
        "effective_grad_accum": fallback["gradient_accumulation_steps"],
        "effective_max_seq_length": fallback["max_seq_length"],
        "warnings": [],
    }

    try:
        import torch
    except ImportError:
        info["warnings"].append("PyTorch is not installed; GPU checks unavailable.")
        return info

    if not torch.cuda.is_available():
        info["warnings"].append(
            "No CUDA GPU detected. In Colab, select Runtime -> Change runtime type -> T4 GPU (Free tier; NEVER enable paid GPU/billing)."
        )
        return info

    props = torch.cuda.get_device_properties(0)
    vram_gb = round(props.total_memory / (1024 ** 3), 2)
    major, minor = torch.cuda.get_device_capability(0)
    supports_ampere = (major, minor) >= (8, 0)

    info["cuda_available"] = True
    info["gpu_name"] = props.name
    info["vram_gb"] = vram_gb
    info["compute_capability"] = f"{major}.{minor}"
    # Enforce T4 safe precision policy per hardware_target (fp16=True, bf16=False on Colab T4)
    info["supports_bf16"] = bool(t4_target.get("supports_bf16", False)) and supports_ampere
    info["supports_flash_attention_2"] = bool(t4_target.get("supports_flash_attention_2", False)) and supports_ampere

    if vram_gb <= 15.5:
        info["warnings"].append(
            f"Detected <=15.5GB GPU ({props.name}, {vram_gb} GB, sm_{major}{minor}). "
            "Applying Free T4 safeguards: fp16=True, bf16=False, gradient_checkpointing=True, paged_adamw_8bit."
        )
    if vram_gb < 14.5:
        info["use_t4_oom_fallback"] = True
        info["effective_batch_size"] = fallback["batch_size"]
        info["effective_grad_accum"] = fallback["gradient_accumulation_steps"]
        info["effective_max_seq_length"] = fallback["max_seq_length"]
        info["warnings"].append(
            f"VRAM ({vram_gb} GB) < 14.5 GB -> auto-switching to t4_oom_fallback: "
            f"batch_size={fallback['batch_size']}, grad_accum={fallback['gradient_accumulation_steps']}, "
            f"max_seq_length={fallback['max_seq_length']}."
        )
    else:
        info["effective_batch_size"] = base_train["per_device_train_batch_size"]
        info["effective_grad_accum"] = base_train["gradient_accumulation_steps"]
        info["effective_max_seq_length"] = base_train["max_seq_length"]

    if not supports_ampere:
        info["effective_fp16"] = True
        info["effective_bf16"] = False
    return info


GPU_SAFEGUARD = inspect_gpu_and_t4_safeguards(TRAIN_CFG)
for _w in GPU_SAFEGUARD["warnings"]:
    print(f"[T4 Safeguard] {_w}")
print("[Hardware Profile]", GPU_SAFEGUARD)


## Step 2: Run `prepare_dataset.py` & `validate_dataset.py`
## Bước 2: Chạy `prepare_dataset.py` (Chia tập Stratified) & `validate_dataset.py` (Kiểm định)

Executes `scripts/prepare_dataset.py` (stratified split by `category`, excluding any overlap with `datasets/evaluation/eval_benchmark.jsonl`) followed by `scripts/validate_dataset.py --require-processed`.


In [ ]:
import json
import subprocess

raw_file = AI_TRAINING_DIR / PATHS_CFG["local_raw_seed_file"]
eval_file = AI_TRAINING_DIR / PATHS_CFG["local_eval_benchmark_file"]
processed_dir = (AI_TRAINING_DIR / PATHS_CFG["local_train_file"]).parent
outputs_dir = AI_TRAINING_DIR / PATHS_CFG["local_outputs_dir"]

prep_cmd = [
    sys.executable,
    str(AI_TRAINING_DIR / "scripts" / "prepare_dataset.py"),
    "--raw", str(raw_file),
    "--eval-file", str(eval_file),
    "--processed-dir", str(processed_dir),
    "--output-dir", str(outputs_dir),
    "--seed", str(TRAIN_CFG["experiment"]["seed"]),
]
prep_res = subprocess.run(prep_cmd, capture_output=True, text=True, check=True)
print("[prepare_dataset.py stdout]")
print(prep_res.stdout.strip())

val_cmd = [
    sys.executable,
    str(AI_TRAINING_DIR / "scripts" / "validate_dataset.py"),
    "--raw", str(raw_file),
    "--processed-dir", str(processed_dir),
    "--eval-file", str(eval_file),
    "--output-dir", str(outputs_dir),
    "--require-processed",
]
val_res = subprocess.run(val_cmd, capture_output=True, text=True, check=True)
print("\n[validate_dataset.py stdout]")
print(val_res.stdout.strip())


## Step 3: Inspect Quality Report, Category Balance, Safety & Leakage Check
## Bước 3: Kiểm tra Báo cáo Chất lượng, Cân bằng Nhóm Chủ đề, An toàn & Rò rỉ Dữ liệu


In [ ]:
import json

report_path = outputs_dir / "dataset_quality_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

print(f"Overall Passed : {report['passed']}")
print(f"Zero Leakage   : {report['leakage_check']['zero_leakage']}")
print(f"Errors ({len(report['errors'])})     : {report['errors']}")
print(f"Warnings ({len(report['warnings'])})   :")
for w in report["warnings"]:
    print(f"  - {w}")

print("\n--- Dataset Counts & Safety Summary ---")
for name, ds in report["datasets"].items():
    unrev = ds.get("reviewed_counts", {}).get("false", 0)
    pii_cnt = len(ds.get("pii_findings", []))
    safety_cnt = len(ds.get("safety_findings", []))
    print(
        f"{name:15s} | total={ds['total_records']:2d} | single={ds['single_turn_records']:2d} | "
        f"multi={ds['multi_turn_records']:2d} | reviewed=false:{unrev:2d} | pii={pii_cnt} | safety_err={safety_cnt}"
    )

print("\n--- Category Distribution (All 8 Required Categories) ---")
ds_names = list(report["datasets"].keys())
header = f"{'category':38s} | " + " | ".join(f"{n:>14s}" for n in ds_names)
print(header)
print("-" * len(header))
categories = list(next(iter(report["datasets"].values()))["category_counts"].keys())
for cat in categories:
    vals = " | ".join(f"{report['datasets'][n]['category_counts'].get(cat, 0):14d}" for n in ds_names)
    print(f"{cat:38s} | {vals}")

print("\n--- Split & Benchmark Leakage Pair Checks ---")
for pair_name, stats in report["leakage_check"]["pair_checks"].items():
    print(f"  {pair_name:35s} -> {stats}")


## Step 4: Preview Sample Dialogues & Optional Qwen3 Token Length Distribution
## Bước 4: Xem trước Hội thoại Mẫu & (Tùy chọn) Thống kê Độ dài Token Qwen3

- **Default (`RUN_TOKENIZER_DISTRIBUTION = False`)**: uses a fast offline word/character token estimate without touching Hugging Face Hub.
- **On Colab (`RUN_TOKENIZER_DISTRIBUTION = True`)**: downloads only the lightweight `Qwen/Qwen3-4B-Instruct-2507` tokenizer (~several MB, **zero model weights**) to verify exact token counts against `max_seq_length` (`1024` standard / `768` T4 OOM fallback).


In [ ]:
from validate_dataset import load_jsonl

RUN_TOKENIZER_DISTRIBUTION = False  # Set True on Colab to inspect exact Qwen3 tokenizer lengths (downloads tokenizer files only, NOT model weights)

train_recs, _ = load_jsonl(processed_dir / "train.jsonl")
val_recs, _ = load_jsonl(processed_dir / "val.jsonl")
test_recs, _ = load_jsonl(processed_dir / "test.jsonl")
eval_recs, _ = load_jsonl(eval_file)

print(f"Loaded splits -> train={len(train_recs)}, val={len(val_recs)}, test={len(test_recs)}, eval_benchmark={len(eval_recs)}")
if train_recs:
    sample = train_recs[0]
    print(f"\n[Sample Train Dialogue: {sample['id']} | category={sample['category']} | safety={sample['safety_level']} | reviewed={sample['reviewed']}]")
    for m in sample["messages"]:
        preview = m["content"] if len(m["content"]) <= 180 else m["content"][:180] + "..."
        print(f"  [{m['role']}] {preview}")

all_split_recs = [("train", train_recs), ("val", val_recs), ("test", test_recs)]
max_seq_std = TRAIN_CFG["training"]["max_seq_length"]
max_seq_safe = TRAIN_CFG["t4_oom_fallback"]["max_seq_length"]

if RUN_TOKENIZER_DISTRIBUTION:
    from transformers import AutoTokenizer
    tok_id = MODEL_CFG["primary_model"]["id"]
    print(f"\nLoading tokenizer only (no model weights): {tok_id}")
    tokenizer = AutoTokenizer.from_pretrained(tok_id)
    for split_name, recs in all_split_recs:
        lengths = []
        for r in recs:
            rendered = tokenizer.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)
            lengths.append(len(tokenizer(rendered, add_special_tokens=False)["input_ids"]))
        lengths_sorted = sorted(lengths)
        p95 = lengths_sorted[min(len(lengths_sorted) - 1, int(0.95 * len(lengths_sorted)))]
        over_std = sum(1 for x in lengths if x > max_seq_std)
        over_safe = sum(1 for x in lengths if x > max_seq_safe)
        print(
            f"[Exact Qwen3 Tokens] {split_name:5s}: n={len(lengths):2d}, min={min(lengths):3d}, "
            f"avg={sum(lengths)/len(lengths):.1f}, p95={p95:3d}, max={max(lengths):3d} | "
            f">{max_seq_safe}={over_safe}, >{max_seq_std}={over_std}"
        )
else:
    print("\n[Offline Token Estimate] (Set RUN_TOKENIZER_DISTRIBUTION=True in Colab for exact Qwen3 BPE counts)")
    for split_name, recs in all_split_recs:
        est_tokens = [
            int(sum(len(m["content"].split()) for m in r["messages"]) * 1.65) + 24
            for r in recs
        ]
        print(
            f"[Approx Tokens] {split_name:5s}: n={len(est_tokens):2d}, min={min(est_tokens):3d}, "
            f"avg={sum(est_tokens)/len(est_tokens):.1f}, max={max(est_tokens):3d} "
            f"(target max_seq_length: std={max_seq_std}, t4_fallback={max_seq_safe})"
        )


## Step 5: Persist Processed Splits & Quality Reports to Google Drive
## Bước 5: Sao lưu Tập dữ liệu Đã xử lý & Báo cáo lên Google Drive

Saves `train.jsonl`, `val.jsonl`, `test.jsonl`, `eval_benchmark.jsonl`, and `dataset_quality_report.{json,md}` into `/content/drive/MyDrive/mimamori_companion_ai/` for disconnect resilience across notebooks 03, 04, and 05.


In [ ]:
files_to_sync_data = [
    (processed_dir / "train.jsonl", DRIVE_PROCESSED_DATA_DIR / "train.jsonl"),
    (processed_dir / "val.jsonl", DRIVE_PROCESSED_DATA_DIR / "val.jsonl"),
    (processed_dir / "test.jsonl", DRIVE_PROCESSED_DATA_DIR / "test.jsonl"),
    (eval_file, DRIVE_PROCESSED_DATA_DIR / "eval_benchmark.jsonl"),
]
files_to_sync_reports = [
    (outputs_dir / "dataset_quality_report.json", DRIVE_EVAL_OUTPUTS_DIR / "dataset_quality_report.json"),
    (outputs_dir / "dataset_quality_report.md", DRIVE_EVAL_OUTPUTS_DIR / "dataset_quality_report.md"),
]

for src, dst in files_to_sync_data + files_to_sync_reports:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
    print(f"[Persisted] {dst} ({dst.stat().st_size} bytes)")

print("\n[Done] Dataset preparation and Drive backup completed cleanly.")
